In [1]:
chunks = [
    "Artificial intelligence is transforming healthcare through predictive analytics.",
    "Machine learning algorithms improve recommendation systems in e-commerce.",
    "Natural language processing enables chatbots to understand human language.",
    "Cloud computing provides scalable infrastructure for modern applications.",
    "Cybersecurity protects systems from unauthorized access and cyber threats.",
    "Data science combines statistics, programming, and domain expertise.",
    "Deep learning models require large datasets for effective training.",
    "Vector databases are optimized for similarity search using embeddings.",
    "FAISS is a popular library for approximate nearest neighbor search.",
    "HNSW constructs a graph structure for efficient vector retrieval.",
    "Python is widely used for machine learning and data analysis.",
    "SQL databases store structured data using relational tables.",
    "NoSQL databases provide flexibility for semi-structured information.",
    "Docker simplifies application deployment using containers.",
    "Kubernetes automates container orchestration in cloud environments.",
    "Large language models generate human-like text responses.",
    "Semantic search retrieves documents based on meaning rather than keywords.",
    "Embeddings convert text into dense numerical vectors.",
    "Retrieval augmented generation combines search with language models.",
    "Transformers revolutionized natural language processing tasks.",
    "Computer vision enables machines to interpret digital images.",
    "Speech recognition converts spoken language into text.",
    "Reinforcement learning trains agents through rewards and penalties.",
    "Distributed systems improve scalability and fault tolerance.",
    "Graph databases represent relationships between entities efficiently.",
    "Apache Spark processes large-scale data in distributed environments.",
    "TensorFlow is a framework for deep learning applications.",
    "PyTorch is widely used for research in artificial intelligence.",
    "Neural networks consist of interconnected layers of artificial neurons.",
    "Principal component analysis reduces data dimensionality.",
    "Clustering groups similar data points without labels.",
    "Classification predicts discrete categories for input samples.",
    "Regression estimates continuous numerical values.",
    "Decision trees provide interpretable machine learning models.",
    "Random forests combine multiple decision trees for better accuracy.",
    "Gradient boosting builds strong predictive models iteratively.",
    "Support vector machines classify data using optimal hyperplanes.",
    "Cross-validation estimates model performance on unseen data.",
    "Feature engineering improves machine learning performance.",
    "Hyperparameter tuning optimizes model configurations.",
    "Cosine similarity measures the angle between vectors.",
    "Euclidean distance measures straight-line distance in space.",
    "Approximate nearest neighbor search trades accuracy for speed.",
    "Indexing accelerates search operations on large datasets.",
    "Caching reduces response times by storing frequent results.",
    "Load balancing distributes traffic across multiple servers.",
    "Microservices divide applications into independent services.",
    "REST APIs enable communication between software systems.",
    "GraphQL provides flexible querying for APIs.",
    "Data warehouses support analytical workloads."
]
len(chunks)

50

# Flat Index 

In [30]:
from langchain_ollama import OllamaEmbeddings
import faiss
import pickle
import numpy as np 

class FaissFlat:
    def __init__(self,model="nomic-embed-text",
                index_path = "./Flat_Index/flat_index.index",
                chunk_path = "./Flat_Index/flat_chunks.pkl"):
        
        self.emb_model = OllamaEmbeddings(model=model)
        self.index_path = index_path
        self.chunk_path = chunk_path

    def get_embeddings(self, chunks):
        if len(chunks) == 0:
            print("Chunks are empty lits")
            return 0 
        chunk_embeddings = self.emb_model.embed_documents(chunks)
        chunk_embeddings = np.array(chunk_embeddings, dtype="float32")
        return chunk_embeddings

    def save_embeddings(self,chunks,search_method="IP"):
        
        chunk_embeddings = self.get_embeddings(chunks)

        dim = chunk_embeddings.shape[1]
        if search_method == "IP":
            self.index = faiss.IndexFlatIP(dim)  # Dot(Inner) Product  - Angle
        else:
            self.index = faiss.IndexFlatL2(dim)  # Euclidean Distance - Distance 
        
        # Add data to index 
        self.index.add(chunk_embeddings)
        
        # Save Index to file 
        faiss.write_index(self.index, self.index_path)
        print("Index Saved in :", self.index_path)
        
        with open(self.chunk_path, "wb") as f:
            pickle.dump(chunks, f)
        print("Chunks Saved in :", self.chunk_path)

    def load_embeddings(self):
        # Load index and Chunks 
        self.index = faiss.read_index(self.index_path)
        print("Loaded index from : ", self.index_path)
        
        with open(self.chunk_path, "rb") as fl:
            self.documents = pickle.load(fl)
        print("Loaded chunks from : ", self.chunk_path)


    def search(self, query, k=2):
        query_embeddings = self.get_embeddings(query)
        print("Query Embeddings shape", query_embeddings.shape)
        distances, doc_index = self.index.search(query_embeddings, k=k)
        print(distances[0], doc_index[0])

        results = [self.documents[ind] for ind in doc_index[0]]
        return results

In [33]:
faiss_obj = FaissFlat()
# faiss_obj.save_embeddings(chunks)  # Create embeddings for all chunks and save the results 
faiss_obj.load_embeddings()
print("Indexing Method is : ", faiss_obj.index.__class__.__name__)

Loaded index from :  ./Flat_Index/flat_index.index
Loaded chunks from :  ./Flat_Index/flat_chunks.pkl
Indexing Method is :  IndexFlatL2


In [34]:
query = "artificial Intelligence  usages ?"
results = faiss_obj.search(query)
print(results)

Query Embeddings shape (1, 768)
[0.5730351 0.6116474] [ 0 27]
['Artificial intelligence is transforming healthcare through predictive analytics.', 'PyTorch is widely used for research in artificial intelligence.']


In [35]:
query = "Support Vector Machine usages"
results = faiss_obj.search(query)
print(results)

Query Embeddings shape (1, 768)
[0.61572754 0.77398545] [36 10]
['Support vector machines classify data using optimal hyperplanes.', 'Python is widely used for machine learning and data analysis.']


# HNSW 

In [36]:
from langchain_ollama import OllamaEmbeddings
import faiss
import pickle
import numpy as np 

class FaissHNSW:
    def __init__(self,model="nomic-embed-text",
                index_path = "./Hnsw_Index/flat_index.index",
                chunk_path = "./Hnsw_Index/flat_chunks.pkl"):
        
        self.emb_model = OllamaEmbeddings(model=model)
        self.index_path = index_path
        self.chunk_path = chunk_path

    def get_embeddings(self, chunks):
        if len(chunks) == 0:
            print("Chunks are empty lits")
            return 0 
        chunk_embeddings = self.emb_model.embed_documents(chunks)
        chunk_embeddings = np.array(chunk_embeddings, dtype="float32")
        return chunk_embeddings

    def save_embeddings(self,chunks,search_method="IP"):
        
        chunk_embeddings = self.get_embeddings(chunks)
        dim = chunk_embeddings.shape[1]

        M = 6   # How many nieghbour chunks should be connected
        efConstruction = 10  # Beighbours to check during graph construction 
        efSearch = 8  # how many neighbours to serach during retreival 
        
        if search_method == "IP":
            self.index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)  # Dot(Inner) Product  - Angle
        else:
            self.index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_L2)  # Euclidean Distance - Distance 

        # Set Other Paramaters 
        self.index.hnsw.efConstruction = efConstruction
        self.index.hnsw.efSearch = efSearch
        
        # Add data to index 
        self.index.add(chunk_embeddings)
        
        # Save Index to file 
        faiss.write_index(self.index, self.index_path)
        print("HNSW Index Saved in :", self.index_path)
        
        with open(self.chunk_path, "wb") as f:
            pickle.dump(chunks, f)
        print("Chunks Saved in :", self.chunk_path)

    def load_embeddings(self):
        # Load index and Chunks 
        self.index = faiss.read_index(self.index_path)
        print("Loaded index from : ", self.index_path)
        
        with open(self.chunk_path, "rb") as fl:
            self.documents = pickle.load(fl)
        print("Loaded chunks from : ", self.chunk_path)


    def search(self, query, k=2):
        query_embeddings = self.get_embeddings(query)
        print("Query Embeddings shape", query_embeddings.shape)
        distances, doc_index = self.index.search(query_embeddings, k=k)
        print(distances[0], doc_index[0])
        results = [self.documents[ind] for ind in doc_index[0]]
        return results

In [40]:
faiss_obj_hnsw = FaissHNSW()
# faiss_obj_hnsw.save_embeddings(chunks)  # Create embeddings for all chunks and save the results 
faiss_obj_hnsw.load_embeddings()
print("Indexing Method is : ", faiss_obj_hnsw.index.__class__.__name__)

Loaded index from :  ./Hnsw_Index/flat_index.index
Loaded chunks from :  ./Hnsw_Index/flat_chunks.pkl
Indexing Method is :  IndexHNSWFlat


In [41]:
query = "Support Vector Machine usages"
results = faiss_obj_hnsw.search(query)
print(results)

Query Embeddings shape (1, 768)
[0.69213575 0.61300683] [36 10]
['Support vector machines classify data using optimal hyperplanes.', 'Python is widely used for machine learning and data analysis.']
